In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import math
import matplotlib.pyplot as plt

In [ ]:
def target_pdf(y):
    # ---------------------------------------------------------
    # STEP 1: AUTOREGRESSIVE DALITZ KINEMATICS (s_tot = 100)
    # ---------------------------------------------------------
    sig_y = torch.sigmoid(y)
    
    # s12 ranges from 0.1 to 99.0 (Leaves at least 1.0 GeV for the other particles)
    s12 = 0.1 + 98.9 * sig_y[:, 0]
    
    # s23 width is dynamically calculated. 
    # Max available space is (100.0 total - s12 - 0.1 min for s13) = 99.9 - s12
    # Because s12 is capped at 99.0, the width is ALWAYS at least +0.9 (Safe from zero!)
    s23_width = 99.9 - s12 
    s23 = 0.1 + s23_width * sig_y[:, 1]
    
    # s13 is completely determined by energy conservation
    s13 = 100.0 - s12 - s23
    
    # The 3 Quantum Angles
    cos_theta = 2.0 * sig_y[:, 2] - 1.0
    sin_theta_sq = 1.0 - cos_theta**2
    phi = 2.0 * math.pi * sig_y[:, 3]
    alpha = 2.0 * math.pi * sig_y[:, 4]
    
    # ---------------------------------------------------------
    # STEP 2: THE AUTOREGRESSIVE JACOBIAN
    # ---------------------------------------------------------
    log_jac_0 = math.log(98.9) + F.logsigmoid(y[:, 0]) + F.logsigmoid(-y[:, 0])
    
    # Now perfectly safe! s23_width is mathematically guaranteed to be positive.
    log_jac_1 = torch.log(s23_width) + F.logsigmoid(y[:, 1]) + F.logsigmoid(-y[:, 1])
    
    log_jac_2 = math.log(2.0) + F.logsigmoid(y[:, 2]) + F.logsigmoid(-y[:, 2])
    log_jac_3 = math.log(2.0 * math.pi) + F.logsigmoid(y[:, 3]) + F.logsigmoid(-y[:, 3])
    log_jac_4 = math.log(2.0 * math.pi) + F.logsigmoid(y[:, 4]) + F.logsigmoid(-y[:, 4])
    
    log_bounding_jac = log_jac_0 + log_jac_1 + log_jac_2 + log_jac_3 + log_jac_4
    
    # ---------------------------------------------------------
    # STEP 3: THE 5 BREIT-WIGNER RESONANCES
    # ---------------------------------------------------------
    denom_1 = (s13 - 40.0)**2 + 16.0
    re_1 = (s13 - 40.0) / denom_1
    im_1 = -4.0 / denom_1
    
    denom_2 = (s12 - 30.0)**2 + 4.0
    re_2 = (s12 - 30.0) / denom_2
    im_2 = -2.0 / denom_2
    
    denom_3 = (s12 - 33.0)**2 + 2.25
    re_3 = ((s12 - 33.0) * torch.cos(phi) + 1.5 * torch.sin(phi)) / denom_3
    im_3 = ((s12 - 33.0) * torch.sin(phi) - 1.5 * torch.cos(phi)) / denom_3
    
    denom_4 = (s23 - 15.0)**2 + 1.0
    re_4 = sin_theta_sq * ((s23 - 15.0) / denom_4)
    im_4 = sin_theta_sq * (-1.0 / denom_4)
    
    denom_5 = (s23 - 80.0)**2 + 1.0
    re_5 = (torch.cos(alpha)**2) * ((s23 - 80.0) / denom_5)
    im_5 = (torch.cos(alpha)**2) * (-1.0 / denom_5)
    
    # ---------------------------------------------------------
    # STEP 4: QUANTUM SUPERPOSITION
    # ---------------------------------------------------------
    total_re = re_1 + re_2 + re_3 + re_4 + re_5
    total_im = im_1 + im_2 + im_3 + im_4 + im_5
    
    matrix_element_sq = total_re**2 + total_im**2
    log_f_phys = torch.log(matrix_element_sq + 1e-10)
    
    return log_f_phys + log_bounding_jac

In [ ]:
# ==========================================
# 2. THE MACHINE LEARNING ARCHITECTURE
# ==========================================
class AffineCouplingLayer(nn.Module):
    def __init__(self, dim, hidden_dim, mask):
        super().__init__()
        self.mask = mask
        self.mlp = nn.Sequential(
            nn.Linear(dim, hidden_dim),
            nn.LeakyReLU(0.01),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LeakyReLU(0.01),
            nn.Linear(hidden_dim, dim * 2) 
        )
        
        # --- THE FIX: Identity Initialization ---
        # Force the final layer to output exactly 0.0 at the start of training.
        # This guarantees s=0 and t=0, preventing the space from exploding.
        nn.init.zeros_(self.mlp[-1].weight)
        nn.init.zeros_(self.mlp[-1].bias)

    def forward(self, z):
        z_A = z * self.mask
        s, t = self.mlp(z_A).chunk(2, dim=-1)
        
        # We can keep the clamp as an extra safety net
        s = torch.clamp(s, min=-5.0, max=5.0) 
        
        inv_mask = 1.0 - self.mask
        s = s * inv_mask
        t = t * inv_mask
        
        x = z_A + inv_mask * (z * torch.exp(s) + t)
        log_det = s.sum(dim=-1)
        return x, log_det

class NormalizingFlow(nn.Module):
    def __init__(self, dim=5, num_layers=16, hidden_dim=128):
        super().__init__()
        self.layers = nn.ModuleList()
        for i in range(num_layers):
            
            # --- THE FIX: DYNAMIC N-DIMENSIONAL MASK ---
            mask = torch.zeros(dim)
            if i % 2 == 0:
                mask[::2] = 1.0  # Even layers mask: [1, 0, 1, 0, 1, 0, 1, 0]
            else:
                mask[1::2] = 1.0 # Odd layers mask:  [0, 1, 0, 1, 0, 1, 0, 1]
                
            self.layers.append(AffineCouplingLayer(dim, hidden_dim, mask))

    def forward(self, z):
        log_det_total = torch.zeros(z.shape[0], device=z.device)
        for layer in self.layers:
            # We must ensure the mask is on the same device as z (e.g., CPU or GPU)
            layer.mask = layer.mask.to(z.device) 
            z, log_det = layer(z)
            log_det_total += log_det
        return z, log_det_total

In [ ]:
def train_flow():
    dim = 5
    flow = NormalizingFlow(dim=dim, num_layers=16, hidden_dim=128)
    
    # Start with a slightly higher learning rate since the scheduler will shrink it
    optimizer = optim.Adam(flow.parameters(), lr=2e-3)
    
    epochs = 5000 
    batch_size = 8192 # Increased batch size for smoother gradients with a larger network
    
    # --- UPGRADE 1: Cosine Annealing Scheduler ---
    # Smoothly reduces the learning rate to 0 following a cosine curve
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    
    beta_start = 0.05       
    beta_end = 1.0          
    anneal_epochs = 3000
    
    # --- NEW: EARLY STOPPING TRACKERS ---
    best_loss = float('inf')
    patience_counter = 0
    patience_limit = 300  # How many epochs we tolerate no improvement
    
    print("Starting Training with Early Stopping Monitor...")
    
    for epoch in range(epochs):
        optimizer.zero_grad()
        
        beta = beta_start + (beta_end - beta_start) * min(1.0, epoch / anneal_epochs)
            
        z = torch.randn(batch_size, dim)
        log_p_z = -0.5 * (z**2).sum(dim=-1) - (dim / 2.0) * math.log(2 * math.pi)
        
        x, log_det = flow(z)
        log_q_x = log_p_z - log_det
        log_f_x = target_pdf(x)
        
        loss = (log_q_x - beta * log_f_x).mean()
        loss.backward()
        
        # --- UPGRADE 2: Gradient Clipping ---
        # Prevents exploding gradients (NaNs) by capping the maximum gradient vector length at 1.0
        nn.utils.clip_grad_norm_(flow.parameters(), max_norm=1.0)
        
        optimizer.step()
        scheduler.step() # Tell the scheduler to update the learning rate
        
        current_loss = loss.item()
        current_lr = scheduler.get_last_lr()[0]
        
        if epoch % 400 == 0:
            print(f"Epoch {epoch:4d} | Beta: {beta:.3f} | LR: {current_lr:.5f} | Loss: {current_loss:.4f}")
            
        # =========================================================
        # THE EARLY STOPPING MONITOR
        # =========================================================
        # Rule 1: Only start tracking AFTER the physics is fully sharpened
        if epoch > anneal_epochs:
            
            # Rule 2: Check if the loss genuinely improved
            if current_loss < best_loss - 1e-4:
                best_loss = current_loss
                patience_counter = 0  # Reset the clock!
            else:
                patience_counter += 1 # Tick the clock
                
            # Rule 3: Pull the plug if we hit the limit
            if patience_counter >= patience_limit:
                print(f"\n[EARLY STOPPING TRIGGERED]")
                print(f"Loss completely flatlined for {patience_limit} epochs.")
                print(f"Training successfully terminated early at Epoch {epoch} to save time.")
                break # This instantly shatters the for-loop

    return flow

In [ ]:
# ==========================================
# 4. UPGRADED PLOTTING (Physical Variables)
# ==========================================
def integrate_and_plot(flow, num_samples=30000000): 
    print("\nStarting Physics Integration...")
    dim = 5
    flow.eval()
    with torch.no_grad():
        z = torch.randn(num_samples, dim)
        log_p_z = -0.5 * (z**2).sum(dim=-1) - (dim / 2.0) * math.log(2 * math.pi)
        
        y, log_det = flow(z)
        log_q_y = log_p_z - log_det
        log_f_y = target_pdf(y)
        
        weights = torch.exp(log_f_y - log_q_y)
        
        integral = weights.mean().item()
        variance = weights.var().item()
        error = math.sqrt(variance / num_samples)
        
        print(f"Calculated Integral : {integral:.6f} +/- {error:.6f}")
        print(f"Variance (sigma^2)  : {variance:.6f}")
        
        # --- THE FIX: MAP TO 0 -> 10 GeV ---
        # --- MAP TO DALITZ KINEMATICS ---
        sig_y0 = torch.sigmoid(y[:, 0])
        sig_y1 = torch.sigmoid(y[:, 1])
        
        s12 = 0.1 + 99.8 * sig_y0
        s23_max = 100.0 - s12
        s23 = 0.1 + (s23_max - 0.2) * sig_y1
        
        x0_np = s12.numpy()
        x1_np = s23.numpy()
    
    # --- NEW CAMERA AND LABELS ---
    plt.figure(figsize=(8, 8))
    # Zoom camera out to the 100x100 Dalitz Space
    plt.hist2d(x0_np, x1_np, bins=200, cmap='magma', range=[[0, 100], [0, 100]])
    plt.colorbar(label='Generated Density')
    
    plt.title("5-Resonance 5D Dalitz Manifold (s_tot = 100)")
    plt.xlabel("$s_{12}$ (Invariant Mass Sq)")
    plt.ylabel("$s_{23}$ (Invariant Mass Sq)")
    plt.show()

In [ ]:
trained_model = train_flow()

In [ ]:
integrate_and_plot(trained_model)